# DeepEval MCP Metrics - Excel Trace Evaluation

This notebook only evaluates MCP trace data from an Excel workbook. The MCP server and trace generator run outside this notebook. The Excel file is the handoff artifact.

## 1. Install Dependencies

In [ ]:
%pip install deepeval==4.0.7 mcp==1.28.1 openai==1.99.5 pandas==3.0.3 openpyxl==3.1.5


## 2. Imports

In [ ]:
from pathlib import Path

import pandas as pd

from deepeval.metrics.mcp.mcp_task_completion import MCPTaskCompletionMetric
from deepeval.metrics.mcp.multi_turn_mcp_use_metric import MultiTurnMCPUseMetric
from deepeval.metrics.mcp_use_metric.mcp_use_metric import MCPUseMetric
from deepeval.models.llms.azure_model import AzureOpenAIModel
from deepeval.test_case import ConversationalTestCase, LLMTestCase, MCPServer, MCPToolCall, Turn
from mcp.types import CallToolResult, TextContent, Tool

## 3. Judge Model Credentials

Set the judge model credentials before reading or evaluating the Excel trace.

In [ ]:
from urllib.parse import urlparse

# Fill these before running metric cells. Do not leave any PASTE_* placeholders.
AZURE_OPENAI_API_KEY = "PASTE_AZURE_OPENAI_API_KEY_HERE"
AZURE_OPENAI_ENDPOINT = ""  # Example: https://your-resource.openai.azure.com
AZURE_OPENAI_DEPLOYMENT_NAME = "PASTE_AZURE_OPENAI_DEPLOYMENT_NAME_HERE"
AZURE_OPENAI_MODEL_NAME = "gpt-5"
AZURE_OPENAI_API_VERSION = "PASTE_AZURE_OPENAI_API_VERSION_HERE"

AZURE_OPENAI_COST_PER_INPUT_TOKEN = 0.0
AZURE_OPENAI_COST_PER_OUTPUT_TOKEN = 0.0


def validate_azure_openai_credentials():
    values = {
        "AZURE_OPENAI_API_KEY": AZURE_OPENAI_API_KEY,
        "AZURE_OPENAI_ENDPOINT": AZURE_OPENAI_ENDPOINT,
        "AZURE_OPENAI_DEPLOYMENT_NAME": AZURE_OPENAI_DEPLOYMENT_NAME,
        "AZURE_OPENAI_API_VERSION": AZURE_OPENAI_API_VERSION,
    }
    missing_or_placeholder = [
        name
        for name, value in values.items()
        if not str(value).strip() or "PASTE_" in str(value)
    ]
    if missing_or_placeholder:
        raise ValueError(
            "Set real Azure OpenAI values before running evaluation metrics. Missing or placeholder fields: "
            + ", ".join(missing_or_placeholder)
        )

    endpoint = AZURE_OPENAI_ENDPOINT.strip().rstrip("/")
    parsed = urlparse(endpoint)
    if parsed.scheme != "https" or not parsed.netloc:
        raise ValueError(
            "AZURE_OPENAI_ENDPOINT must be a full HTTPS URL, for example: "
            "https://your-resource.openai.azure.com"
        )
    if " " in parsed.netloc or "PASTE" in parsed.netloc:
        raise ValueError(f"AZURE_OPENAI_ENDPOINT host is invalid: {parsed.netloc!r}")
    return endpoint


AZURE_OPENAI_ENDPOINT = validate_azure_openai_credentials()

judge_model = AzureOpenAIModel(
    model=AZURE_OPENAI_MODEL_NAME,
    deployment_name=AZURE_OPENAI_DEPLOYMENT_NAME,
    api_key=AZURE_OPENAI_API_KEY,
    base_url=AZURE_OPENAI_ENDPOINT,
    api_version=AZURE_OPENAI_API_VERSION,
    cost_per_input_token=AZURE_OPENAI_COST_PER_INPUT_TOKEN,
    cost_per_output_token=AZURE_OPENAI_COST_PER_OUTPUT_TOKEN,
)


## 4. Excel File

Run the next cell to upload `Live_Dummy_MCP_Traces.xlsx` in Colab. If you are running locally and the file is already beside this notebook, the same cell will use that local file instead.

In [ ]:
TRACE_FILE = Path("Live_Dummy_MCP_Traces.xlsx")

try:
    from google.colab import files

    print("Upload Live_Dummy_MCP_Traces.xlsx")
    uploaded = files.upload()

    if uploaded:
        uploaded_name = next(iter(uploaded.keys()))
        TRACE_FILE = Path(uploaded_name)
except ImportError:
    print("Not running in Colab. Using local Excel file path.")

if not TRACE_FILE.exists():
    raise FileNotFoundError(
        "Live_Dummy_MCP_Traces.xlsx was not found. Upload the generated Excel file or place it beside this notebook."
    )

TRACE_FILE

## 5. Read Generated Sheets

In [ ]:
input_cases = pd.read_excel(TRACE_FILE, sheet_name="input_cases")
tools_definition = pd.read_excel(TRACE_FILE, sheet_name="tools_definition")
deepeval_objects = pd.read_excel(TRACE_FILE, sheet_name="deepeval_objects")
support_data = pd.read_excel(TRACE_FILE, sheet_name="support_data")

required_input_columns = {"case_id", "expected_tools_called", "actual_tools_called", "run_status"}
missing_columns = required_input_columns - set(input_cases.columns)
if missing_columns:
    raise ValueError(f"input_cases sheet is missing columns: {sorted(missing_columns)}")

required_deepeval_columns = {
    "case_id",
    "dummy_mcp_server_object",
    "llm_test_case_object",
    "conversational_test_case_object",
}
missing_deepeval_columns = required_deepeval_columns - set(deepeval_objects.columns)
if missing_deepeval_columns:
    raise ValueError(f"deepeval_objects sheet is missing columns: {sorted(missing_deepeval_columns)}")

input_cases


## 6. Inspect Tool Definitions

In [ ]:
pd.set_option("display.max_colwidth", 3000)

tools_definition[["tool_name", "description", "deepeval_tool_object"]]


## 7. Inspect Generated DeepEval Objects

In [ ]:
deepeval_objects[["case_id", "dummy_mcp_server_object", "llm_test_case_object", "conversational_test_case_object"]]


## 8. Load MCP Server Object From Excel

In [ ]:
server_source = deepeval_objects["dummy_mcp_server_object"].dropna().astype(str).str.strip()
if server_source.empty:
    raise ValueError("deepeval_objects has no dummy_mcp_server_object snippet to load.")

server_namespace = {"MCPServer": MCPServer, "Tool": Tool}
exec(server_source.iloc[0], {"__builtins__": {}, **server_namespace}, server_namespace)
dummy_support_server = server_namespace["dummy_support_server"]

dummy_support_server


## 9. Test Case Shapes

`MCPUseMetric` evaluates single-turn `LLMTestCase` objects. `MultiTurnMCPUseMetric` and `MCPTaskCompletionMetric` evaluate the `ConversationalTestCase` objects already written in the Excel trace.

The notebook uses the syntaxed DeepEval objects from the `deepeval_objects` sheet directly.

## 10. Use DeepEval Test Cases From Excel

In [ ]:
def first_non_empty(series):
    for value in series:
        if pd.notna(value) and str(value).strip():
            return str(value)
    return ""


def expected_label_from_status(run_status):
    run_status = str(run_status).strip()
    if run_status and run_status not in {"completed", "step_limit"}:
        return run_status
    return "Positive"


def evaluate_deepeval_object(source, case_id, column_name):
    if pd.isna(source) or not str(source).strip():
        raise ValueError(f"Missing {column_name} for case {case_id}")
    try:
        return eval(str(source), deepeval_eval_globals, {})
    except Exception as exc:
        raise ValueError(f"Could not load {column_name} for case {case_id}: {exc}") from exc


def mcp_tool_result_value(tool):
    structured_content = getattr(tool.result, "structuredContent", None) or {}
    if isinstance(structured_content, dict) and "result" in structured_content:
        return structured_content["result"]
    return structured_content


from deepeval.metrics.mcp.schema import Task


def get_tasks_from_excel_conversations(self, unit_interactions):
    tasks = []

    for unit_interaction in unit_interactions:
        if len(unit_interaction) < 2:
            continue
        if not any(turn._mcp_interaction for turn in unit_interaction[1:]):
            continue

        user_messages = ""
        for turn in unit_interaction:
            if turn.role == "user":
                user_messages += turn.content + "\n"
            else:
                break

        new_task = Task(task=user_messages, steps_taken=[])

        for turn in unit_interaction[1:]:
            if turn._mcp_interaction:
                mcp_interaction = "Tools called by agent: \n"
                if turn.mcp_tools_called is not None:
                    for tool in turn.mcp_tools_called:
                        mcp_interaction += (
                            f"\n<Tool Called>\n"
                            f"\n**This does not appear to user**\n"
                            f"Name: {tool.name}\n"
                            f"Args: {tool.args}\n"
                            f"Result: \n{mcp_tool_result_value(tool)}\n"
                            f"</Tool Called>\n"
                        )
                new_task.steps_taken.append(mcp_interaction)

                if str(turn.content).strip():
                    new_task.steps_taken.append("Agent's response to user: \n" + turn.content)
            else:
                new_task.steps_taken.append("Agent's response to user: \n" + turn.content)

        tasks.append(new_task)

    return tasks


MultiTurnMCPUseMetric._get_tasks = get_tasks_from_excel_conversations
MCPTaskCompletionMetric._get_tasks = get_tasks_from_excel_conversations


input_case_metadata = []
for case_id, case_rows in input_cases.fillna("").groupby("case_id", sort=False):
    input_case_metadata.append(
        {
            "case_id": str(case_id),
            "expected_label": expected_label_from_status(first_non_empty(case_rows["run_status"])),
            "expected_tools_called": first_non_empty(case_rows["expected_tools_called"]),
            "actual_tools_called": first_non_empty(case_rows["actual_tools_called"]),
        }
    )
case_metadata_by_id = {row["case_id"]: row for row in input_case_metadata}

deepeval_eval_globals = {
    "__builtins__": {},
    "LLMTestCase": LLMTestCase,
    "ConversationalTestCase": ConversationalTestCase,
    "MCPServer": MCPServer,
    "MCPToolCall": MCPToolCall,
    "Turn": Turn,
    "CallToolResult": CallToolResult,
    "TextContent": TextContent,
    "Tool": Tool,
    "dummy_support_server": dummy_support_server,
}

test_cases = []
conversational_test_cases = []
evaluation_case_rows = []

for _, object_row in deepeval_objects.fillna("").iterrows():
    case_id = str(object_row["case_id"]).strip()
    if not case_id:
        continue

    llm_test_case = evaluate_deepeval_object(object_row["llm_test_case_object"], case_id, "llm_test_case_object")
    conversational_test_case = evaluate_deepeval_object(
        object_row["conversational_test_case_object"],
        case_id,
        "conversational_test_case_object",
    )

    test_cases.append(llm_test_case)
    conversational_test_cases.append(conversational_test_case)

    metadata = case_metadata_by_id.get(case_id, {})
    evaluation_case_rows.append(
        {
            "case_id": case_id,
            "expected_label": metadata.get("expected_label", "Positive"),
            "tool_call_count": len(llm_test_case.mcp_tools_called or []),
            "expected_tools_called": metadata.get("expected_tools_called", ""),
            "actual_tools_called": metadata.get("actual_tools_called", ""),
        }
    )

from deepeval.metrics.utils import get_unit_interactions

conversation_debug_rows = []
for conversational_case, case_row in zip(conversational_test_cases, evaluation_case_rows):
    unit_interactions = get_unit_interactions(conversational_case.turns)
    metric_tasks = get_tasks_from_excel_conversations(None, unit_interactions)
    conversation_debug_rows.append(
        {
            "case_id": case_row["case_id"],
            "turn_count": len(conversational_case.turns),
            "unit_interaction_count": len(unit_interactions),
            "metric_task_count": len(metric_tasks),
            "tool_call_count": case_row["tool_call_count"],
        }
    )

conversation_debug = pd.DataFrame(conversation_debug_rows)
empty_task_cases = conversation_debug[conversation_debug["metric_task_count"] == 0]
if not empty_task_cases.empty:
    raise ValueError(
        "Multi-turn metrics would extract empty Tasks for these cases: "
        + ", ".join(empty_task_cases["case_id"].astype(str).tolist())
    )

len(test_cases), len(conversational_test_cases), pd.DataFrame(evaluation_case_rows), conversation_debug


## 11. Run MCPUseMetric

In [ ]:
mcp_use_rows = []

for test_case, case_row in zip(test_cases, evaluation_case_rows):
    metric = MCPUseMetric(model=judge_model, threshold=0.5, include_reason=True, async_mode=False, verbose_mode=True)
    score = metric.measure(test_case)
    mcp_use_rows.append(
        {
            "case_id": case_row["case_id"],
            "expected_label": case_row["expected_label"],
            "metric": "MCPUseMetric",
            "score": round(score, 3),
            "success": metric.is_successful(),
            "reason": metric.reason,
        }
    )

mcp_use_report = pd.DataFrame(mcp_use_rows)
mcp_use_report

## 12. Run MultiTurnMCPUseMetric

In [ ]:
multi_turn_use_rows = []

for conversational_case, case_row in zip(conversational_test_cases, evaluation_case_rows):
    metric = MultiTurnMCPUseMetric(model=judge_model, threshold=0.5, include_reason=True, async_mode=False, verbose_mode=True)
    score = metric.measure(conversational_case)
    multi_turn_use_rows.append(
        {
            "case_id": case_row["case_id"],
            "expected_label": case_row["expected_label"],
            "metric": "MultiTurnMCPUseMetric",
            "score": round(score, 3),
            "success": metric.is_successful(),
            "reason": metric.reason,
        }
    )

multi_turn_use_report = pd.DataFrame(multi_turn_use_rows)
multi_turn_use_report

## 13. Run MCPTaskCompletionMetric

In [ ]:
task_completion_rows = []

for conversational_case, case_row in zip(conversational_test_cases, evaluation_case_rows):
    metric = MCPTaskCompletionMetric(model=judge_model, threshold=0.5, include_reason=True, async_mode=False, verbose_mode=True)
    score = metric.measure(conversational_case)
    task_completion_rows.append(
        {
            "case_id": case_row["case_id"],
            "expected_label": case_row["expected_label"],
            "metric": "MCPTaskCompletionMetric",
            "score": round(score, 3),
            "success": metric.is_successful(),
            "reason": metric.reason,
        }
    )

mcp_task_completion_report = pd.DataFrame(task_completion_rows)
mcp_task_completion_report

## 14. Final Report

In [ ]:
final_report = pd.concat(
    [mcp_use_report, multi_turn_use_report, mcp_task_completion_report],
    ignore_index=True,
)

final_report

## 15. Export Final Report

This cell writes the metric results to Excel. In Colab, it also starts a browser download.

In [ ]:
REPORT_FILE = Path("DeepEval_MCP_Final_Report.xlsx")

with pd.ExcelWriter(REPORT_FILE, engine="openpyxl") as writer:
    final_report.to_excel(writer, sheet_name="final_report", index=False)
    mcp_use_report.to_excel(writer, sheet_name="mcp_use_metric", index=False)
    multi_turn_use_report.to_excel(writer, sheet_name="multi_turn_mcp_use", index=False)
    mcp_task_completion_report.to_excel(writer, sheet_name="mcp_task_completion", index=False)

print(f"Saved report to {REPORT_FILE}")

try:
    from google.colab import files

    files.download(str(REPORT_FILE))
except ImportError:
    from IPython.display import FileLink, display

    display(FileLink(str(REPORT_FILE)))